# Randomize yearly datasets into game trials

Assign the 13 years from 2004–2016 to block 2 in shuffled order. Independently shuffle the 11 years from 2005–2015 and split them across blocks 1 and 3. The same assignment is used for each matching `items`, `pf`, and `pf_indices` file.

In [ ]:
from pathlib import Path
import csv
import random
import shutil

sub_id = 0

# Paths are relative to this notebook/repository root.
DATA_DIR = Path("data")
ITEMS_DIR = DATA_DIR / "items"
PF_DIR = DATA_DIR / "unbiased_pf"
OUTPUT_DIR = DATA_DIR / "game_data"

YEARS = list(range(2004, 2017))
YEARS_2 = list(range(2005, 2016))
BLOCK_SIZES = {1: 6, 2: 13, 3: 5}
SEED = 1123 + sub_id  # Change this value to create a different reproducible assignment.


def source_files(year):
    """Return the three source files associated with one year."""
    return {
        "items": ITEMS_DIR / f"items_{year}.csv",
        "pf": PF_DIR / f"pf_{year}.csv",
        "pf_indices": PF_DIR / f"pf_indices_{year}.csv",
    }


# Validate all inputs before writing anything.
missing = [
    path
    for year in YEARS
    for path in source_files(year).values()
    if not path.is_file()
]
if missing:
    raise FileNotFoundError(
        "Missing source files:\n" + "\n".join(str(path) for path in missing)
    )

rng = random.Random(SEED)
block_2_years = YEARS.copy()
block_1_and_3_years = YEARS_2.copy()
rng.shuffle(block_2_years)
rng.shuffle(block_1_and_3_years)

assignments = {
    1: block_1_and_3_years[: BLOCK_SIZES[1]],
    2: block_2_years,
    3: block_1_and_3_years[BLOCK_SIZES[1] :],
}
print(assignments)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = []

for block in BLOCK_SIZES:
    for trial, year in enumerate(assignments[block], start=1):
        destinations = {
            "items": OUTPUT_DIR / f"block_{block}_trial_{trial}_{year}.csv",
            "pf": OUTPUT_DIR / f"pf_block_{block}_trial_{trial}_{year}.csv",
            "pf_indices": OUTPUT_DIR / f"pf_indices_block_{block}_trial_{trial}_{year}.csv",
        }

        for kind, source in source_files(year).items():
            shutil.copy2(source, destinations[kind])

        manifest.append({"block": block, "trial": trial, "year": year})

# Save the randomization separately so it is easy to inspect and reproduce.
manifest_path = OUTPUT_DIR / "trial_assignments.csv"
with manifest_path.open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["block", "trial", "year"])
    writer.writeheader()
    writer.writerows(manifest)

print(f"Copied {len(manifest) * 3} files to {OUTPUT_DIR}")
print(f"Assignment manifest: {manifest_path}")
manifest

{1: [2009, 2014, 2006, 2007, 2011, 2013], 2: [2009, 2005, 2010, 2015, 2011, 2006, 2007, 2013, 2004, 2014, 2012, 2016, 2008], 3: [2012, 2010, 2008, 2015, 2005]}
Copied 72 files to data/game_data
Assignment manifest: data/game_data/trial_assignments.csv


[{'block': 1, 'trial': 1, 'year': 2009},
 {'block': 1, 'trial': 2, 'year': 2014},
 {'block': 1, 'trial': 3, 'year': 2006},
 {'block': 1, 'trial': 4, 'year': 2007},
 {'block': 1, 'trial': 5, 'year': 2011},
 {'block': 1, 'trial': 6, 'year': 2013},
 {'block': 2, 'trial': 1, 'year': 2009},
 {'block': 2, 'trial': 2, 'year': 2005},
 {'block': 2, 'trial': 3, 'year': 2010},
 {'block': 2, 'trial': 4, 'year': 2015},
 {'block': 2, 'trial': 5, 'year': 2011},
 {'block': 2, 'trial': 6, 'year': 2006},
 {'block': 2, 'trial': 7, 'year': 2007},
 {'block': 2, 'trial': 8, 'year': 2013},
 {'block': 2, 'trial': 9, 'year': 2004},
 {'block': 2, 'trial': 10, 'year': 2014},
 {'block': 2, 'trial': 11, 'year': 2012},
 {'block': 2, 'trial': 12, 'year': 2016},
 {'block': 2, 'trial': 13, 'year': 2008},
 {'block': 3, 'trial': 1, 'year': 2012},
 {'block': 3, 'trial': 2, 'year': 2010},
 {'block': 3, 'trial': 3, 'year': 2008},
 {'block': 3, 'trial': 4, 'year': 2015},
 {'block': 3, 'trial': 5, 'year': 2005}]

In [2]:
# Randomize all 17 years (2000–2016) into block 2 without repetition.
from pathlib import Path
import csv
import random
import shutil

new_sub_id = 0
new_data_dir = Path("data")
new_items_dir = new_data_dir / "items"
new_pf_dir = new_data_dir / "unbiased_pf"
new_output_dir = new_data_dir / "game_data"
new_years = list(range(2000, 2017))
new_seed = 1123 + new_sub_id  # Change this value for another reproducible order.


def new_source_files(year):
    """Return the matching items, pf, and pf_indices files for one year."""
    return {
        "items": new_items_dir / f"items_{year}.csv",
        "pf": new_pf_dir / f"pf_{year}.csv",
        "pf_indices": new_pf_dir / f"pf_indices_{year}.csv",
    }


def new_destination_files(trial, year):
    """Return the block 2 output paths for one trial."""
    trial_name = f"block_2_trial_{trial}_{year}"
    return {
        "items": new_output_dir / f"{trial_name}.csv",
        "pf": new_output_dir / f"pf_{trial_name}.csv",
        "pf_indices": new_output_dir / f"pf_indices_{trial_name}.csv",
    }


# Validate all 51 source files before writing output.
new_missing = [
    path
    for year in new_years
    for path in new_source_files(year).values()
    if not path.is_file()
]
if new_missing:
    raise FileNotFoundError(
        "Missing source files:\n" + "\n".join(str(path) for path in new_missing)
    )

new_rng = random.Random(new_seed)
randomized_years = new_years.copy()
new_rng.shuffle(randomized_years)

assert len(randomized_years) == 17
assert len(set(randomized_years)) == 17

new_output_dir.mkdir(parents=True, exist_ok=True)
new_manifest = []

for trial, year in enumerate(randomized_years, start=1):
    sources = new_source_files(year)
    destinations = new_destination_files(trial, year)

    for kind, source in sources.items():
        shutil.copy2(source, destinations[kind])

    new_manifest.append({"block": 2, "trial": trial, "year": year})

new_manifest_path = new_output_dir / "trial_assignments.csv"
with new_manifest_path.open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["block", "trial", "year"])
    writer.writeheader()
    writer.writerows(new_manifest)

print(f"Copied {len(new_manifest) * 3} files to {new_output_dir}")
print(f"Assignment manifest: {new_manifest_path}")
new_manifest

Copied 51 files to data/game_data
Assignment manifest: data/game_data/trial_assignments.csv


[{'block': 2, 'trial': 1, 'year': 2005},
 {'block': 2, 'trial': 2, 'year': 2002},
 {'block': 2, 'trial': 3, 'year': 2011},
 {'block': 2, 'trial': 4, 'year': 2006},
 {'block': 2, 'trial': 5, 'year': 2003},
 {'block': 2, 'trial': 6, 'year': 2001},
 {'block': 2, 'trial': 7, 'year': 2015},
 {'block': 2, 'trial': 8, 'year': 2010},
 {'block': 2, 'trial': 9, 'year': 2004},
 {'block': 2, 'trial': 10, 'year': 2007},
 {'block': 2, 'trial': 11, 'year': 2012},
 {'block': 2, 'trial': 12, 'year': 2013},
 {'block': 2, 'trial': 13, 'year': 2000},
 {'block': 2, 'trial': 14, 'year': 2014},
 {'block': 2, 'trial': 15, 'year': 2016},
 {'block': 2, 'trial': 16, 'year': 2009},
 {'block': 2, 'trial': 17, 'year': 2008}]

In [6]:
import pandas as pd
import numpy as np

pf = pd.read_csv("data/game_data/pf_block_1_trial_2_2007.csv")

In [7]:
print(np.max(pf, axis=0))
print(np.min(pf, axis=0))

PTS    175.037071
TRB    116.221292
STL     17.942813
BLK     24.026517
FG%    595.900000
dtype: float64
PTS    105.582382
TRB     61.415401
STL      8.149073
BLK      6.351332
FG%    460.700000
dtype: float64


In [10]:
ranges = pd.read_csv("data/game_data/block_1_trial_2_2007_single_solution.csv")
print(ranges)
first_row = ranges.drop(columns="SALARY").iloc[0].to_numpy(dtype=float)
print(first_row)

                                                 PTS  \
0                                         202.895541   
1    [30, 45, 35, 178, 111, 302, 246, 251, 308, 311]   
2                                          56.843783   
3  [153, 168, 299, 189, 227, 230, 331, 180, 271, 19]   

                                                 TRB  \
0                                         123.098337   
1    [239, 271, 47, 163, 93, 25, 145, 157, 230, 299]   
2                                          23.303449   
3  [228, 277, 315, 294, 170, 323, 125, 122, 253, ...   

                                                STL  \
0                                         21.360585   
1   [160, 140, 251, 18, 325, 307, 41, 304, 45, 219]   
2                                          3.312151   
3  [152, 1, 203, 296, 238, 120, 210, 290, 168, 141]   

                                                 BLK  \
0                                          29.985836   
1   [120, 168, 25, 234, 299, 66, 152, 161, 250, 13